# Data mix v3 UCE-brain embeddings (v2026-09 vocabulary)

Runs a model trained with the **v2026-09 gene vocabulary** (266,403 tokens, 12 species keys) on an h5ad file and plots a UMAP of the cell embeddings:

* `KuanP/brain-uce-pilot-mix-v3` — data mix v3: 13-species designed mixture, 131,072 steps at global batch 512 (weights upload pending);
* `KuanP/brain-uce-pilot-mix-v2` — uniform v2026-09 mix, 262,144 steps at global batch 256.

Both share the architecture of the earlier models (8 layers, d_model 512, 4 heads) but not their tokenisation: the chromosome-token offset, sentence length and special ids are read from the checkpoint (`config.yaml` / `config.json`) by `uce_brain.inference.load_cell_sentence_params` instead of being defaulted. Species keys are NCBI binomials (`homo_sapiens`, `macaca_mulatta`, ...); the legacy spellings (`human`, `mouse`, ...) and common names (`macaque`, `marmoset`, ...) are resolved automatically. Gene symbols are matched case-insensitively against the upper-case vocabulary, so mouse/rat sentence-case symbols work without preprocessing.

Input expectations: **raw UMI counts** in `X` (or `raw`), gene symbols in `var_names` or a `var` column (Ensembl ids are resolved through `feature_name` and friends).

In [ ]:
# --- Parameters -------------------------------------------------------------
# MODEL: Hugging Face repo id, or a local checkpoint directory holding config.json +
# model.safetensors with the training run's config.yaml next to them. The v3 weights
# are still being uploaded; until they land, the local sibling run (identical
# vocabulary and architecture) works, e.g.
#   "/dfs/ssd/project/uce-brain/hf_upload/brain-uce-pilot-mix-v2"   (staged Hub layout)
#   "/dfs/ssd/project/uce-brain/runs/brain_multi_v2026_09_noeval_262144steps/2026-09-20_23-48-44"
MODEL = "KuanP/brain-uce-pilot-mix-v3"

# Gene dictionary. None = the copy shipped inside the model repo (vocab/...), else this
# repository's gene_data/all_species_gene_dict_v2026-09.json. A raw run directory has
# no vocab/, so point this at the gene_data copy when MODEL is a run directory.
GENE_MAPPING_PATH = None

# Any h5ad with raw UMI counts. Default: Allen HMBA whole-brain atlas v0.5, macaque
# (raw counts; taxonomy labels in obs: neighborhood / class / subclass).
H5AD_PATH = "/dfs/ssd/project/uce-brain/data/eval_data/hmba_v05/macaque/hmba_wb_v0.5_macaca_mulatta_nonoverlap.h5ad"
SPECIES = "macaca_mulatta"              # gene-dict key or alias: "human", "mouse", "macaque", "marmoset", ...
LABEL_KEYS = ["class", "neighborhood"]  # obs columns to colour the UMAP by (missing ones are skipped)

MAX_CELLS = 5000       # None = every cell
BATCH_SIZE = 32
NUM_WORKERS = 4
SUBSAMPLE_SEED = 0

In [ ]:
from pathlib import Path

import numpy as np
import scanpy as sc
import torch

import uce_brain
from uce_brain.data import load_gene_mapping, read_h5ad_subsampled
from uce_brain.inference import (
    embed_adata,
    find_gene_mapping,
    load_cell_sentence_params,
    load_model,
    resolve_checkpoint,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Load the model, its tokenisation parameters and the gene dictionary

In [ ]:
ckpt = resolve_checkpoint(MODEL)          # downloads the repo when MODEL is a Hub id
params = load_cell_sentence_params(ckpt)  # pad_length, chrom_token_offset, ... from the checkpoint
model = load_model(ckpt, device=device)   # fp32 weights; bf16 autocast is used at embedding time

gene_mapping_path = (
    GENE_MAPPING_PATH
    or find_gene_mapping(ckpt)
    or Path(uce_brain.__file__).resolve().parents[2] / "gene_data" / "all_species_gene_dict_v2026-09.json"
)
gene_mapping = load_gene_mapping(str(gene_mapping_path))
print(f"vocab_size={model.config.vocab_size}  d_model={model.config.d_model}  layers={model.config.num_layers}")
print("cell-sentence params:", params.as_kwargs())
print("species keys:", sorted(gene_mapping))

## 2. Load the cells

In [ ]:
if MAX_CELLS is None:
    adata = sc.read_h5ad(H5AD_PATH)
else:
    try:
        # Reads only the sampled rows of a CSR-encoded X (fast on multi-GB files).
        adata = read_h5ad_subsampled(H5AD_PATH, n_cells=MAX_CELLS, seed=SUBSAMPLE_SEED)
    except ValueError:  # dense X: full read, then subsample
        adata = sc.read_h5ad(H5AD_PATH)
        sc.pp.sample(adata, n=min(MAX_CELLS, adata.n_obs), rng=SUBSAMPLE_SEED)
print(adata.shape, "X:", type(adata.X).__name__, adata.X.dtype)
print("label columns present:", [c for c in LABEL_KEYS if c in adata.obs.columns])

## 3. Extract cell embeddings

In [ ]:
adata.obsm["X_uce"] = embed_adata(
    model, adata, gene_mapping, params,
    species=SPECIES,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    mask_prop=0.0,          # no gene masking for embeddings
    case_insensitive=True,  # vocabulary keys are upper-case; matches sentence-case symbols too
)
emb = adata.obsm["X_uce"]
norms = np.linalg.norm(emb, axis=1)
print(f"embeddings: {emb.shape}  finite: {np.isfinite(emb).all()}  L2 norm: {norms.min():.4f} .. {norms.max():.4f}")

## 4. UMAP

In [ ]:
sc.pp.neighbors(adata, use_rep="X_uce", n_neighbors=15, metric="cosine")
sc.tl.umap(adata, min_dist=0.3)

In [ ]:
color = [c for c in LABEL_KEYS if c in adata.obs.columns]
if color:
    sc.pl.umap(adata, color=color, ncols=2, frameon=False, wspace=0.5, legend_fontsize=6)
else:
    print("none of", LABEL_KEYS, "in adata.obs; plotting an unlabeled UMAP")
    sc.pl.umap(adata, frameon=False)

## 5. Quick label-transfer check

Cosine kNN accuracy (5-fold, on the loaded cells) against the majority-class baseline. A working model is far above the baseline. Note that this does not catch tokenisation mistakes: a wrong chromosome-token offset still gives unit-norm embeddings and only a slightly lower kNN score, which is why the parameters are taken from the checkpoint (`load_cell_sentence_params`) rather than defaulted.

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier

for key in color:
    y = adata.obs[key].astype(str).to_numpy(dtype=object)  # plain numpy array (sklearn cannot index pyarrow-backed strings)
    pred = cross_val_predict(
        KNeighborsClassifier(n_neighbors=15, metric="cosine", weights="distance"),
        emb, y, cv=KFold(n_splits=5, shuffle=True, random_state=0),
    )
    majority = np.unique(y, return_counts=True)[1].max() / len(y)
    print(f"{key}: kNN accuracy {np.mean(pred == y):.3f}  (majority class {majority:.3f}, {len(set(y))} classes)")